<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Introduction" data-toc-modified-id="Introduction-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Introduction</a></span></li><li><span><a href="#Prepare-the-data" data-toc-modified-id="Prepare-the-data-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Prepare the data</a></span><ul class="toc-item"><li><span><a href="#Load-packages" data-toc-modified-id="Load-packages-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Load packages</a></span></li><li><span><a href="#explore-the-data-directory" data-toc-modified-id="explore-the-data-directory-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>explore the data directory</a></span></li><li><span><a href="#Load-the-rating-data-to-pandas-dataframe" data-toc-modified-id="Load-the-rating-data-to-pandas-dataframe-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Load the rating data to pandas dataframe</a></span></li><li><span><a href="#Load-the-item-data" data-toc-modified-id="Load-the-item-data-2.4"><span class="toc-item-num">2.4&nbsp;&nbsp;</span>Load the item data</a></span></li><li><span><a href="#Convert-data-into-user-item-rating-matrix-and-user-item-binary-rating-matrix" data-toc-modified-id="Convert-data-into-user-item-rating-matrix-and-user-item-binary-rating-matrix-2.5"><span class="toc-item-num">2.5&nbsp;&nbsp;</span>Convert data into user-item rating matrix and user-item binary rating matrix</a></span><ul class="toc-item"><li><span><a href="#check-the-number-of-users-and-items" data-toc-modified-id="check-the-number-of-users-and-items-2.5.1"><span class="toc-item-num">2.5.1&nbsp;&nbsp;</span>check the number of users and items</a></span></li><li><span><a href="#Construct-the-user-time-rating-matrix" data-toc-modified-id="Construct-the-user-time-rating-matrix-2.5.2"><span class="toc-item-num">2.5.2&nbsp;&nbsp;</span>Construct the user-time rating matrix</a></span></li></ul></li></ul></li><li><span><a href="#Filtering" data-toc-modified-id="Filtering-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Filtering</a></span><ul class="toc-item"><li><span><a href="#Load-the-cosine_similarity-package-to-calculate-the-pairwise-cosine-similarity-between-users" data-toc-modified-id="Load-the-cosine_similarity-package-to-calculate-the-pairwise-cosine-similarity-between-users-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Load the cosine_similarity package to calculate the pairwise cosine similarity between users</a></span></li><li><span><a href="#calculate-user-similarity" data-toc-modified-id="calculate-user-similarity-3.2"><span class="toc-item-num">3.2&nbsp;&nbsp;</span>calculate user similarity</a></span></li><li><span><a href="#Create-a-dataframe-to-store-the-prediction-of-user-ratings-on-un-rated-items" data-toc-modified-id="Create-a-dataframe-to-store-the-prediction-of-user-ratings-on-un-rated-items-3.3"><span class="toc-item-num">3.3&nbsp;&nbsp;</span>Create a dataframe to store the prediction of user ratings on un-rated items</a></span></li><li><span><a href="#calculate-the-value-in-data_pre" data-toc-modified-id="calculate-the-value-in-data_pre-3.4"><span class="toc-item-num">3.4&nbsp;&nbsp;</span>calculate the value in data_pre</a></span></li></ul></li></ul></div>

# User-based Collaborative Filtering with Python

## Introduction
Learn how to do user-based Collaborative Filtering with Python

## Prepare the data

### Load packages

In [5]:
import numpy as np
import pandas as pd

### explore the data directory

Move into the data directory

In [1]:
cd ml-100k/

/Users/jayarajkumarayyappan/Pythonworks/ml-100k


List the files in the ml-100k folder

In [2]:
ls

README        u.info        u1.test       u4.base       ua.test
allbut.pl*    u.item        u2.base       u4.test       ub.base
mku.sh*       u.occupation  u2.test       u5.base       ub.test
u.data        u.user        u3.base       u5.test
u.genre       u1.base       u3.test       ua.base


Print the first 10 lines in u.data:

In [3]:
!head u.data

196	242	3	881250949
186	302	3	891717742
22	377	1	878887116
244	51	2	880606923
166	346	1	886397596
298	474	4	884182806
115	265	2	881171488
253	465	5	891628467
305	451	3	886324817
6	86	3	883603013


### Load the rating data to pandas dataframe

In [6]:
names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('u.data', sep='\t', names=names)
df.head()

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


### Load the item data

In [7]:
i_cols = ['movie id', 'movie title' ,'release date','video release date', 'IMDb URL', 'unknown', 'Action', 'Adventure',
 'Animation', 'Children\'s', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy',
 'Film-Noir', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
items = pd.read_csv('u.item', sep='|', names=i_cols, encoding='latin-1')

items.head()

,movie id,movie title,release date,video release date,IMDb URL,unknown,Action,Adventure,Animation,Children's,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Toy%20Story%2...,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,2,GoldenEye (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?GoldenEye%20(...,0,1,1,0,0,...,0,0,0,0,0,0,0,1,0,0
2,3,Four Rooms (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Four%20Rooms%...,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
3,4,Get Shorty (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Get%20Shorty%...,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Copycat (1995),01-Jan-1995,NaN,http://us.imdb.com/M/title-exact?Copycat%20(1995),0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


### Convert data into user-item rating matrix and user-item binary rating matrix

#### check the number of users and items

In [8]:
n_users = df.user_id.unique().shape[0]
n_items = df.item_id.unique().shape[0]
print(f"{n_users} users")
print(f"{n_items} items")

943 users
1682 items


#### Construct the user-time rating matrix

In [9]:
ratings = np.zeros((n_users, n_items))

In [10]:
for row in df.itertuples():
    ratings[row[1]-1, row[2]-1] = row[3]
print(ratings)

[[5. 3. 4. ... 0. 0. 0.]
 [4. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [5. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 5. 0. ... 0. 0. 0.]]


Convert the 'ratings' to a dataframe

In [11]:
ratings_df = pd.DataFrame(ratings)

In [12]:
ratings_df.head()

,0,1,2,3,4,5,6,7,8,9,...,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681
0,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Filtering

### Load the cosine_similarity package to calculate the pairwise cosine similarity between users

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

### calculate user similarity

In [14]:
userSimilarity = cosine_similarity(ratings, ratings)

convert the numpy array to dataframe

In [15]:
data_ubs_fast = pd.DataFrame(userSimilarity)

In [16]:
data_ubs_fast.head()

,0,1,2,3,4,5,6,7,8,9,...,933,934,935,936,937,938,939,940,941,942
0,1.000000,0.166931,0.047460,0.064358,0.378475,0.430239,0.440367,0.319072,0.078138,0.376544,...,0.369527,0.119482,0.274876,0.189705,0.197326,0.118095,0.314072,0.148617,0.179508,0.398175
1,0.166931,1.000000,0.110591,0.178121,0.072979,0.245843,0.107328,0.103344,0.161048,0.159862,...,0.156986,0.307942,0.358789,0.424046,0.319889,0.228583,0.226790,0.161485,0.172268,0.105798
2,0.047460,0.110591,1.000000,0.344151,0.021245,0.072415,0.066137,0.083060,0.061040,0.065151,...,0.031875,0.042753,0.163829,0.069038,0.124245,0.026271,0.161890,0.101243,0.133416,0.026556
3,0.064358,0.178121,0.344151,1.000000,0.031804,0.068044,0.091230,0.188060,0.101284,0.060859,...,0.052107,0.036784,0.133115,0.193471,0.146058,0.030138,0.196858,0.152041,0.170086,0.058752
4,0.378475,0.072979,0.021245,0.031804,1.000000,0.237286,0.373600,0.248930,0.056847,0.201427,...,0.338794,0.080580,0.094924,0.079779,0.148607,0.071459,0.239955,0.139595,0.152497,0.313941


In [17]:
data_ubs_fast

,0,1,2,3,4,5,6,7,8,9,...,933,934,935,936,937,938,939,940,941,942
0,1.000000,0.166931,0.047460,0.064358,0.378475,0.430239,0.440367,0.319072,0.078138,0.376544,...,0.369527,0.119482,0.274876,0.189705,0.197326,0.118095,0.314072,0.148617,0.179508,0.398175
1,0.166931,1.000000,0.110591,0.178121,0.072979,0.245843,0.107328,0.103344,0.161048,0.159862,...,0.156986,0.307942,0.358789,0.424046,0.319889,0.228583,0.226790,0.161485,0.172268,0.105798
2,0.047460,0.110591,1.000000,0.344151,0.021245,0.072415,0.066137,0.083060,0.061040,0.065151,...,0.031875,0.042753,0.163829,0.069038,0.124245,0.026271,0.161890,0.101243,0.133416,0.026556
3,0.064358,0.178121,0.344151,1.000000,0.031804,0.068044,0.091230,0.188060,0.101284,0.060859,...,0.052107,0.036784,0.133115,0.193471,0.146058,0.030138,0.196858,0.152041,0.170086,0.058752
4,0.378475,0.072979,0.021245,0.031804,1.000000,0.237286,0.373600,0.248930,0.056847,0.201427,...,0.338794,0.080580,0.094924,0.079779,0.148607,0.071459,0.239955,0.139595,0.152497,0.313941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
938,0.118095,0.228583,0.026271,0.030138,0.071459,0.111852,0.107027,0.095898,0.039852,0.071460,...,0.066039,0.431154,0.258021,0.226449,0.432666,1.000000,0.087687,0.180029,0.043264,0.144250
939,0.314072,0.226790,0.161890,0.196858,0.239955,0.352449,0.329925,0.246883,0.120495,0.342961,...,0.327153,0.107024,0.187536,0.181317,0.175158,0.087687,1.000000,0.145152,0.261376,0.241028
940,0.148617,0.161485,0.101243,0.152041,0.139595,0.144446,0.059993,0.146145,0.143245,0.090305,...,0.046952,0.203301,0.288318,0.234211,0.313400,0.180029,0.145152,1.000000,0.101642,0.095120
941,0.179508,0.172268,0.133416,0.170086,0.152497,0.317328,0.282003,0.175322,0.092497,0.212330,...,0.226440,0.073513,0.089588,0.129554,0.099385,0.043264,0.261376,0.101642,1.000000,0.182465


### Create a dataframe to store the prediction of user ratings on un-rated items

In [18]:
data_pre = pd.DataFrame(index=range(0, n_users), 
                        columns=items['movie title'])

### calculate the value in data_pre
> it will take very long time to run, for my mac pro, it takes 1 hour. so only select the first 3 users for the recommendation

In [19]:
# small value, avoid divided by 0
epsilon = 1e-9

# loop through each user i and each items j
# only choose the first 3 users
for i in range(0, 3):
    for j in range(0, len(data_pre.columns)):
        if ratings_df.iloc[i,j] > 0:
            data_pre.iloc[i,j]=0
        else:
            # select users who rated j with a mask
            mask_rated_users = ratings_df[j]>0
            # select k nearest neighbour's IDs and similarities
            top_neighbours = data_ubs_fast[mask_rated_users]\
                                .iloc[:, i]\
                                .sort_values(ascending=False)[1:10]\
                                .index
            top_neighbours_sim = data_ubs_fast[mask_rated_users]\
                                .iloc[:, i]\
                                .sort_values(ascending=False)[1:10]\
                                .values
            # select K nearest neighbour's ratings on j
            neighbours_ratings = ratings_df.iloc[top_neighbours, j]
            # final prediction on j for user i by using weighted sum formula
            data_pre.iloc[i,j] = sum(neighbours_ratings*top_neighbours_sim)/(sum(top_neighbours_sim)+epsilon)
                                

In [21]:
# Create a dataframe to store the final top-6 recommendations
data_recommend = pd.DataFrame(index=data_pre.index, 
                              columns = ['1', '2', '3', '4', '5', '6'])

# Make recommendations to each user by using 'for' 
# loop statement to sort the predictions for each user
for i in range(0, len(data_pre.index)):
    data_recommend.iloc[i, 0:] = data_pre.iloc[i,:]\
                                .sort_values(ascending=False)[0:6]\
                                .index\
                                .transpose()
data_recommend.head()

,1,2,3,4,5,6
0,Prefontaine (1997),"Saint of Fort Washington, The (1993)",Golden Earrings (1947),Anna (1996),Some Mother's Son (1996),Santa with Muscles (1996)
1,"Letter From Death Row, A (1998)",Prefontaine (1997),Star Kid (1997),Little City (1998),The Deadly Cure (1996),Some Mother's Son (1996)
2,Braveheart (1995),Star Kid (1997),Hugo Pool (1997),"Letter From Death Row, A (1998)",Prefontaine (1997),Little City (1998)
3,Toy Story (1995),GoldenEye (1995),Four Rooms (1995),Get Shorty (1995),Copycat (1995),Shanghai Triad (Yao a yao yao dao waipo qiao) ...
4,Toy Story (1995),GoldenEye (1995),Four Rooms (1995),Get Shorty (1995),Copycat (1995),Shanghai Triad (Yao a yao yao dao waipo qiao) ...


In [22]:
data_recommend.head()

,1,2,3,4,5,6
0,Prefontaine (1997),"Saint of Fort Washington, The (1993)",Golden Earrings (1947),Anna (1996),Some Mother's Son (1996),Santa with Muscles (1996)
1,"Letter From Death Row, A (1998)",Prefontaine (1997),Star Kid (1997),Little City (1998),The Deadly Cure (1996),Some Mother's Son (1996)
2,Braveheart (1995),Star Kid (1997),Hugo Pool (1997),"Letter From Death Row, A (1998)",Prefontaine (1997),Little City (1998)
3,Toy Story (1995),GoldenEye (1995),Four Rooms (1995),Get Shorty (1995),Copycat (1995),Shanghai Triad (Yao a yao yao dao waipo qiao) ...
4,Toy Story (1995),GoldenEye (1995),Four Rooms (1995),Get Shorty (1995),Copycat (1995),Shanghai Triad (Yao a yao yao dao waipo qiao) ...
